# Human vs Bot 1 — session analysis (writeup §5)

Ingests **every** session export dropped into `sessions/` ("Download my session" in the game),
replays each one through the **Python** engine (cross-engine validation: the web round must
reproduce *bit-exactly*), runs Bot 1 on the identical stream (common random numbers by
construction — the `stream_id` pins the exogenous events), and compares:

- PnL decomposition (spread captured / adverse selection / inventory cost),
- quote-update counts,
- time-weighted mid-vs-V tracking error,
- re-anchor time after every ≥5-tick jump (how long quotes stayed stale).

**Honest framing:** the current corpus is an n=3 self-experiment by the author (one L1 round,
two L5 rounds). Nothing here generalizes; it demonstrates the instrument. Add sessions to the
folder and re-run.

In [1]:
import json, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "sim"))

from picked_off.engine import run_stream
from picked_off.events import parse_stream, parse_event
from picked_off.params import SimParams
from picked_off.scoring import score_round, value_at
from picked_off.bots.base import run_bot
from picked_off.bots.bot1 import Bot1

SESSIONS = sorted((ROOT / "sessions").glob("*.json"))
STREAMS = ROOT / "web" / "public" / "streams"
print(f"{len(SESSIONS)} session(s) found")

3 session(s) found


In [2]:
def load_round(path):
    """Replay a session through the Python engine + run Bot 1 on its stream."""
    s = json.load(open(path))
    params = SimParams.from_meta(s["meta"]["params"])
    sid = s["meta"]["stream_id"]
    stream = json.load(open(STREAMS / f"{sid}.json"))
    exo = parse_stream(stream["event_stream"], params.round_us, require_opening_quote=False)
    quotes = [parse_event(q) for q in s["player_quotes"]]
    human = run_stream(params, sorted([*quotes, *exo], key=lambda e: e.t_us))
    hd, _ = score_round(human)
    rec = s["pnl_decomposition_half_ticks"]
    exact = (hd.total, hd.spread_captured, hd.adverse_selection, hd.inventory_cost) == (
        rec["total"], rec["spread_captured"], rec["adverse_selection"], rec["inventory_cost"])
    bot = run_bot(params, exo, Bot1())
    bd, _ = score_round(bot)
    return dict(sid=sid, level=s["meta"]["level"], params=params,
                human=human, hd=hd, bot=bot, bd=bd, exact=exact)

ROUNDS = [load_round(p) for p in SESSIONS]
assert all(r["exact"] for r in ROUNDS), "web/python engines disagree!"
print("cross-engine check: every session reproduces BIT-EXACTLY in the Python engine")

cross-engine check: every session reproduces BIT-EXACTLY in the Python engine


In [3]:
print(f"{'stream':<12} {'who':<6} {'total':>8} {'spread':>8} {'adverse':>9} {'invent':>8} {'quotes':>7} {'fills':>6}")
for r in ROUNDS:
    for who, res, d in (("human", r["human"], r["hd"]), ("bot1", r["bot"], r["bd"])):
        print(f"{r['sid']:<12} {who:<6} {d.total/2:>8.1f} {d.spread_captured/2:>8.1f} "
              f"{d.adverse_selection/2:>9.1f} {d.inventory_cost/2:>8.1f} "
              f"{len(res.quotes):>7} {len(res.fills):>6}")

stream       who       total   spread   adverse   invent  quotes  fills
L1-605045    human     481.0    495.0       5.0    -19.0      38     61
L1-605045    bot1      203.0    292.0    -147.0     58.0     177    187
L5-1037333   human     388.0    404.5     -96.5     80.0      16     88
L5-1037333   bot1      196.0    305.0    -168.0     59.0     244    128
L5-1038342   human    -872.0    714.0   -2403.0    817.0      38    207
L5-1038342   bot1       60.0    350.5    -347.5     57.0     242    129


In [4]:
def tracking_error(result, params):
    """Time-weighted mean |quote mid - V|, ticks."""
    qs = result.quotes
    times = sorted({0, *[q.t_us for q in qs], *[j.t_us for j in result.jumps], params.round_us})
    err, qi = 0.0, -1
    for a, b in zip(times, times[1:]):
        while qi + 1 < len(qs) and qs[qi + 1].t_us <= a:
            qi += 1
        v = value_at(params.v0, result.jumps, a)
        err += abs((qs[qi].bid + qs[qi].ask) / 2 - v) * (b - a)
    return err / params.round_us

def reanchor_times(result, params, jump_min=5, close=5):
    """Per |jump| >= jump_min: seconds until |mid - V| <= close ticks (inf if never)."""
    out = []
    qs = result.quotes
    for j in result.jumps:
        if abs(j.size) < jump_min:
            continue
        cur = None
        for q in qs:
            if q.t_us <= j.t_us:
                cur = q
        v = value_at(params.v0, result.jumps, j.t_us)
        if cur and abs((cur.bid + cur.ask) / 2 - v) <= close:
            out.append(0.0)
            continue
        t = float("inf")
        for q in qs:
            if q.t_us <= j.t_us:
                continue
            v_now = value_at(params.v0, result.jumps, q.t_us)
            if abs((q.bid + q.ask) / 2 - v_now) <= close:
                t = (q.t_us - j.t_us) / 1e6
                break
        out.append(t)
    return out

print(f"{'stream':<12} {'who':<6} {'track-err':>10} {'re-anchor times after |J|>=5 (s)'}")
for r in ROUNDS:
    for who, res in (("human", r["human"]), ("bot1", r["bot"])):
        te = tracking_error(res, r["params"])
        ra = ", ".join("never" if t == float("inf") else f"{t:.1f}" for t in reanchor_times(res, r["params"]))
        print(f"{r['sid']:<12} {who:<6} {te:>9.1f}t  [{ra}]")

stream       who     track-err re-anchor times after |J|>=5 (s)
L1-605045    human        5.6t  [5.5, 2.0, 0.0, 0.0, 16.8, 0.0, 1.0, 0.0, 0.9]
L1-605045    bot1         9.1t  [5.3, 1.9, 0.0, 29.3, 23.0, 12.3, 5.0, 4.2, 0.0]
L5-1037333   human        2.7t  [6.1, 0.0]
L5-1037333   bot1         1.9t  [0.3, 0.1]
L5-1038342   human       18.8t  [never, never, never, never, never, never, never, never]
L5-1038342   bot1         3.4t  [1.2, 0.0, 0.8, 0.2, 2.0, 0.7, 0.0, 3.9]


## Findings (n=3 self-experiment — illustration, not evidence)

1. **Cross-engine validation held**: all sessions replay bit-exactly through the Python engine
   — the shared-vector contract works on live human rounds, not just frozen vectors.
2. **L1: the human beat Bot 1** (+481 vs +203 ticks) by quoting wider than regret-free prices
   and harvesting noise — exactly the gate's prediction that naive/wide play is rational at α=0.1.
3. **L5 split 1-1.** The winning round tracked value nearly as well as the bot (2.7 vs 1.9 ticks
   time-weighted error) while monetizing a wider spread. The losing round (−872 vs bot +60) is a
   pure repricing failure: after **all eight** ≥5-tick jumps the human mid *never* returned to
   within 5 ticks of value — informed flow billed the stale quotes for −2403 ticks of adverse
   selection (207 fills!). The bot re-anchored within 0–3.9 s every time.
4. **Style gap:** the human re-quoted 16–38 times/round; the bot ~180–240. The human's edge,
   when it exists, is spread selection; the bot's is relentless recentering. The obvious hybrid
   (human spread, bot mid) is untested — future work.